# 🧠 Data-Centric AI Agent — Training Notebook
**OpenEnv Hackathon 2026** · [Space](https://huggingface.co/spaces/Aswinis-Kumar/data-centric-env) · [GitHub](https://github.com/CelestialWorthyOfHeavenAndEarth/data-centric-env)

Trains **Qwen2.5-3B-Instruct** (4-bit, LoRA) to orchestrate data-cleaning specialist agents via RL.

| Phase | What happens | Time |
|-------|-------------|------|
| SFT warmup | Teaches valid command grammar on ~9k examples | ~20 min |
| GRPO training | Learns strategy via live environment rollouts | ~3–5 hrs |
| Eval + plots | Reward curves, baseline comparison | ~5 min |

**Runtime required:** T4 GPU → `Runtime → Change runtime type → T4 GPU`

> **Quick demo mode:** Set `DEMO_MODE = True` in cell 1 to run a tiny 10-step version in ~10 min.

## ⚙️ Step 1 — Configuration

In [ ]:
# ─── Set DEMO_MODE=True for a quick 10-step judge verification run (~10 min)
DEMO_MODE = False  # ← Change to True for quick verification

REPO     = 'https://github.com/CelestialWorthyOfHeavenAndEarth/data-centric-env.git'
HF_SPACE = 'https://aswinis-kumar-data-centric-env.hf.space'

if DEMO_MODE:
    print('🚀 DEMO MODE — quick 10-step run for judge verification')
    SFT_EPOCHS   = 1
    GRPO_STEPS   = 10
    GRPO_BATCH   = 2
else:
    print('🔬 FULL TRAINING MODE')
    SFT_EPOCHS   = 1
    GRPO_STEPS   = 300   # ~3-5 hrs on T4
    GRPO_BATCH   = 4

print(f'  SFT epochs : {SFT_EPOCHS}')
print(f'  GRPO steps : {GRPO_STEPS}')
print(f'  Batch size : {GRPO_BATCH}')

## 📦 Step 2 — Install Dependencies (~5 min)

In [ ]:
# Step 1: Pin transformers to a version compatible with Colab's torchao
# (transformers >= 4.52 pulls torchao >= 0.9 which requires torch.int1 — not in Colab yet)
!pip install "transformers>=4.44.0,<4.52.0" --quiet

# Step 2: Install Unsloth for the current Colab torch version
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --quiet

# Step 3: Install training + env dependencies
!pip install trl>=0.15.0 datasets>=2.0.0 accelerate>=0.30.0 --quiet
!pip install scikit-learn>=1.3.0 pandas>=2.0.0 numpy matplotlib --quiet
!pip install "openenv-core[core]>=0.2.1" --quiet
!pip install wandb --quiet  # experiment tracking (required)

# Step 4: Verify Unsloth loads correctly
try:
    from unsloth import FastLanguageModel
    print('✅ Unsloth loaded successfully')
except Exception as e:
    print(f'❌ Unsloth import failed: {e}')
    print('   Try: Runtime → Factory reset runtime, then re-run')
    raise

## 📡 Step 2.5 — W&B Experiment Tracking (Required)
Weights & Biases tracks loss, reward curves, and curriculum progression.

Get your free API key: https://wandb.ai/authorize

In [ ]:
import wandb, os

# All runs from this notebook log to this project
os.environ["WANDB_PROJECT"]   = "data-centric-ai-agent"
os.environ["WANDB_LOG_MODEL"] = "false"

# Login with your API key from https://wandb.ai/authorize
wandb.login()

print("\u2705 W&B ready  project: " + os.environ["WANDB_PROJECT"])
print("   Dashboard : https://wandb.ai/")

## 💾 Step 3 — Mount Google Drive (optional, recommended)
Checkpoints save every 50 GRPO steps to Drive. If Colab disconnects, you can resume.

**Skip this cell if you don't want Drive integration.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted')

## 🔁 Step 4 — Clone / Update Repo + Setup

In [ ]:
import os, sys, shutil

WORK_DIR = '/content/data-centric-env'

# Clone or pull latest
if os.path.exists(f'{WORK_DIR}/pyproject.toml'):
    print('Repo exists, pulling latest...')
    !git -C {WORK_DIR} pull origin main
else:
    if os.path.exists(WORK_DIR):
        shutil.rmtree(WORK_DIR)
    !git clone {REPO} {WORK_DIR}

os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)
print('CWD:', os.getcwd())
!git log --oneline -3

# Install as editable package (fixes relative imports)
!pip install -e . --quiet 2>/dev/null || echo 'pip install -e . skipped'

# Symlink output dirs to Drive if mounted
DRIVE_DIR = '/content/drive/MyDrive/data-centric-training'
if os.path.exists('/content/drive/MyDrive'):
    os.makedirs(DRIVE_DIR, exist_ok=True)
    for d in ['data-centric-checkpoints', 'data-centric-adapter', 'logs', 'plots']:
        local  = os.path.join(WORK_DIR, d)
        remote = os.path.join(DRIVE_DIR, d)
        os.makedirs(remote, exist_ok=True)
        if not os.path.exists(local):
            os.symlink(remote, local)
            print(f'  Drive link: {d}')
        else:
            print(f'  Exists: {d}')
    print('Checkpoints will save to Drive')
else:
    for d in ['logs', 'plots']:
        os.makedirs(os.path.join(WORK_DIR, d), exist_ok=True)
    print('Drive not mounted — checkpoints save locally only')

print('Setup complete')

## ✅ Step 5 — Verify All Features Work
Quick smoke test: confirms the environment, all specialist agents, and the rubric system are connected.

In [ ]:
import os, sys
os.chdir('/content/data-centric-env')
sys.path.insert(0, '/content/data-centric-env')

from models import DataCentricAction
from server.data_centric_environment import DataCentricEnvironment

env = DataCentricEnvironment()
obs = env.reset(task='task_1_easy', seed=42)

checks = {}

# 1. query_analyst
obs = env.step(DataCentricAction(message='query_analyst'))
checks['query_analyst'] = 'DIAGNOSIS' in obs.response and 'RECOMMENDED PLAN' in obs.response

# 2. Smarter specialists
obs = env.step(DataCentricAction(message='query_cleaner'))
checks['smarter_specialists'] = any(k in obs.response for k in ['skew', 'Risk:', 'Reason:', '%'])

# 3. Drift detection (apply then check)
obs = env.step(DataCentricAction(message='apply 1'))
checks['drift_detection'] = 'drift' in obs.response.lower() or obs.response != ''

# 4+5. Dual classifier + feature importance
obs = env.step(DataCentricAction(message='validate'))
checks['dual_classifier']    = 'RF Accuracy' in obs.response or 'accuracy' in obs.response.lower()
checks['feature_importance'] = 'Feature importance' in obs.response or 'feature' in obs.response.lower()

print('\n=== Feature Smoke Test ===')
all_pass = True
for name, passed in checks.items():
    status = '✅ PASS' if passed else '❌ FAIL'
    if not passed: all_pass = False
    print(f'  {status}: {name}')

if all_pass:
    print('\n✅ All features verified — ready to train!')
else:
    print('\n⚠️  Some checks failed — check git pull ran correctly')

## 🖥️ Step 6 — Start Environment Server

In [ ]:
import subprocess, time, requests, os, sys
os.chdir('/content/data-centric-env')
sys.path.insert(0, '/content/data-centric-env')

# Kill any existing server on port 8000
!fuser -k 8000/tcp 2>/dev/null || true
time.sleep(2)

server_proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'server.app:app',
     '--host', '0.0.0.0', '--port', '8000', '--log-level', 'warning'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

for i in range(30):
    try:
        if requests.get('http://localhost:8000/health', timeout=2).status_code == 200:
            print(f'✅ Server ready after {i+1}s')
            break
    except:
        time.sleep(1)
else:
    server_proc.terminate()
    out, err = server_proc.communicate()
    print('STDOUT:', out.decode()[-2000:])
    print('STDERR:', err.decode()[-2000:])
    raise RuntimeError('Server failed to start in 30s — see output above')

print('ENV_URL: http://localhost:8000')
os.environ['ENV_URL'] = 'http://localhost:8000'

## 📊 Step 7 — Generate SFT Warmup Data

In [ ]:
import os, time
os.chdir('/content/data-centric-env')

sft_path = 'sft_data.jsonl'
if os.path.exists(sft_path):
    count = sum(1 for _ in open(sft_path))
    age   = (time.time() - os.path.getmtime(sft_path)) / 60
    print(f'sft_data.jsonl: {count} examples, {age:.0f} min old — reusing')
else:
    print('Generating SFT data...')
    !python sft_generator.py
    count = sum(1 for _ in open(sft_path))
    print(f'Generated {count} examples')

## 🤖 Step 8 — Load Model (Qwen2.5-3B-Instruct, 4-bit)

In [ ]:
import os, sys, torch
os.chdir('/content/data-centric-env')
sys.path.insert(0, '/content/data-centric-env')

from unsloth import FastLanguageModel

MODEL_NAME     = 'Qwen/Qwen2.5-3B-Instruct'
MAX_SEQ_LENGTH = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    lora_alpha=32,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

print(f'✅ Model loaded: {MODEL_NAME}')
print(f'   VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB / {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## 🎓 Step 9 — Phase 1: SFT Warmup
Teaches the model to output valid commands before RL starts. Without this, the model outputs free-form text and gets zero reward.

In [ ]:
import os, sys
os.chdir('/content/data-centric-env')
sys.path.insert(0, '/content/data-centric-env')

from train_data_centric import run_sft_warmup

# In demo mode: reduce to 1 pass of 200 examples for speed
if DEMO_MODE:
    import json
    raw = [json.loads(l) for l in open('sft_data.jsonl')][:200]
    with open('sft_data_demo.jsonl', 'w') as f:
        for r in raw: f.write(json.dumps(r) + '\n')
    import shutil
    shutil.copy('sft_data.jsonl', 'sft_data_full.jsonl')
    shutil.copy('sft_data_demo.jsonl', 'sft_data.jsonl')
    print('Demo mode: using 200 SFT examples')

model = run_sft_warmup(model, tokenizer)
print('✅ SFT warmup complete')

# Restore full data if demo
if DEMO_MODE and os.path.exists('sft_data_full.jsonl'):
    import shutil
    shutil.copy('sft_data_full.jsonl', 'sft_data.jsonl')

## 🏋️ Step 10 — Phase 2: GRPO Training
Agent learns *when* to use each command via reinforcement learning against the live environment.

- **Full mode:** ~300 steps, 3–5 hrs on T4. Checkpoints every 50 steps.
- **Demo mode:** 10 steps, ~10 min. Enough to verify the training loop works.

If Colab disconnects, re-run all cells — it auto-resumes from the latest checkpoint.

In [ ]:
import os, sys, glob
os.chdir('/content/data-centric-env')
sys.path.insert(0, '/content/data-centric-env')

# Patch GRPO config for demo mode
if DEMO_MODE:
    import train_data_centric as tdc
    _orig_grpo = tdc.run_grpo_training
    from trl import GRPOConfig
    def _demo_grpo(model, tokenizer, resume_from_checkpoint=None):
        import types
        print(f'DEMO: running {GRPO_STEPS} GRPO steps (batch={GRPO_BATCH})')
        # Temporarily patch the config inside the function
        orig_config_cls = GRPOConfig
        class DemoGRPOConfig(GRPOConfig):
            def __init__(self, *a, **kw):
                kw['max_steps']                   = GRPO_STEPS
                kw['per_device_train_batch_size'] = GRPO_BATCH
                kw['num_generations']             = 2
                super().__init__(*a, **kw)
        import trl
        trl.GRPOConfig = DemoGRPOConfig
        import train_data_centric
        train_data_centric.GRPOConfig = DemoGRPOConfig
        result = _orig_grpo(model, tokenizer, resume_from_checkpoint=resume_from_checkpoint)
        trl.GRPOConfig = orig_config_cls
        train_data_centric.GRPOConfig = orig_config_cls
        return result
    grpo_fn = _demo_grpo
else:
    from train_data_centric import run_grpo_training as grpo_fn

# Auto-detect checkpoint to resume from
ckpt_dir = './data-centric-checkpoints'
resume_from = None
if os.path.exists(ckpt_dir):
    checkpoints = sorted(glob.glob(f'{ckpt_dir}/checkpoint-*'))
    if checkpoints:
        resume_from = checkpoints[-1]
        print(f'Resuming from: {resume_from}')

model = grpo_fn(model, tokenizer, resume_from_checkpoint=resume_from)
print('✅ GRPO training complete')

## 💽 Step 11 — Save Trained Model

In [ ]:
import os, sys
os.chdir('/content/data-centric-env')
sys.path.insert(0, '/content/data-centric-env')

from train_data_centric import save_model
save_model(model, tokenizer)
print('✅ Model saved to ./data-centric-adapter/')

## 📈 Step 12 — Plot Reward Curves

In [ ]:
import os
os.chdir('/content/data-centric-env')

if os.path.exists('logs/training.jsonl'):
    lines = sum(1 for _ in open('logs/training.jsonl'))
    print(f'Training log: {lines} episodes recorded')
    !python plot_rewards.py --log logs/training.jsonl --out plots/
    from IPython.display import Image, display
    for f in ['reward_curve.png', 'success_rate.png', 'accuracy_gain.png', 'curriculum.png']:
        path = f'plots/{f}'
        if os.path.exists(path):
            print(f'\n--- {f} ---')
            display(Image(path))
else:
    print('No training log yet — run Step 10 first')

## 🏆 Step 13 — Evaluate: Trained Agent vs Random vs Heuristic

In [ ]:
import os, json
os.chdir('/content/data-centric-env')

!python eval_data_centric.py

if os.path.exists('eval_results.json'):
    with open('eval_results.json') as f:
        results = json.load(f)
    print('\n=== Eval Results ===')
    for k, v in results.items():
        print(f'  {k}: {v}')

## 📤 Step 14 — Push Results to GitHub
Run this **only after training + eval are done**.

In [ ]:
import os
os.chdir('/content/data-centric-env')

from getpass import getpass
token = getpass('GitHub token (repo write access): ')
repo_url = f'https://{token}@github.com/CelestialWorthyOfHeavenAndEarth/data-centric-env.git'

!git config user.email 'training@colab.run'
!git config user.name 'Colab Training'
!git remote set-url origin {repo_url}
!git add plots/ logs/ eval_results.json 2>/dev/null || true
!git commit -m 'Add training results: reward curves + eval [Colab]'
!git push origin main
print('✅ Results pushed to GitHub')

---
## ✅ Done!

| Output | Location |
|--------|----------|
| Trained LoRA adapter | `./data-centric-adapter/` |
| Training log | `logs/training.jsonl` |
| Reward curves | `plots/*.png` |
| Eval results | `eval_results.json` |

**Links:**  
🤗 [HF Space](https://huggingface.co/spaces/Aswinis-Kumar/data-centric-env) · [Blog Post](https://huggingface.co/blog/Aswinis-Kumar/data-centric-ai-agent) · [GitHub](https://github.com/CelestialWorthyOfHeavenAndEarth/data-centric-env)